# Jump Fade 01 — passive ±1.5σ bracket, take-profit at the pre-move price

Pre-registration: `discussions/Jump_Fade/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/jump_fade.py`.

- **Rule:** at every 5-minute boundary while flat, resting limit buy and sell at $P_0 e^{\mp 1.5\sigma_t}$ for the next 5 minutes. On a fill, the other order is cancelled. Exits: passive take-profit at $P_0$; taker stop at $3\sigma_t$ from $P_0$; taker time stop at 60 minutes.
- **Simulation:** 1-minute Binance USD-M perpetual OHLC with conservative fill ordering. BTC, ETH, SOL, XRP from 2024-10-01. DEV runs to 2025-12-31; TEST is 2026.
- **Splits:** all bars (primary); vol burst (top decile of $\sigma_{1h}/\sigma_t$, DEV threshold); a jump in the last 15 minutes.
- **Verdict (TEST):** G1 gross P&L above the sign-flip null; G2 net at Deribit fees (maker 0, taker 5 bp + 1 bp slippage), daily bootstrap lower bound $>0$, at least 3 of 4 assets.

Environment: `JF01_SYNTHETIC=1`, `JF01_JOBS` (null workers), `JF01_N_NULL` (quick look only).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = str(PROJECT_ROOT / 'src'); sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + (os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
from data_engineering.modeling import jump_fade as jf

SYNTHETIC = os.getenv('JF01_SYNTHETIC', '').strip() == '1'
JOBS = int(os.getenv('JF01_JOBS', str(max(1, (os.cpu_count() or 2) - 1))))
N_NULL = int(os.getenv('JF01_N_NULL', str(jf.N_NULL)))
if SYNTHETIC:
    N_NULL = min(N_NULL, 4); OUT = Path(tempfile.mkdtemp(prefix='jf01_'))
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Jump_Fade' / '01'
OUT.mkdir(parents=True, exist_ok=True)
REGISTERED_RUN = (not SYNTHETIC) and N_NULL == jf.N_NULL
print(jf.selftest()); print('null', N_NULL, 'jobs', JOBS, '| registered' if REGISTERED_RUN else '| NOT registered', '| out', OUT)
print(json.dumps(jf.registration(), indent=1, default=str))

## 1. Data and decision inputs

In [ ]:
if SYNTHETIC:
    MIN = jf.synthetic_minutes(700, seed=4, revert=0.0)
else:
    MIN = {a: jf.load_1m(BIN / f'{a}USDT_um_1m.parquet') for a in jf.ASSETS}
MIN = {a: m[m.index >= jf.DATA_START] for a, m in MIN.items()}
INP = {a: jf.decision_inputs(m) for a, m in MIN.items()}
THR = {a: jf.burst_threshold(i) for a, i in INP.items()}
cov = pd.DataFrame([{'asset': a, 'first': m.index.min(), 'last': m.index.max(), 'minutes': len(m),
                     'missing_share': float(m['close'].isna().mean()), 'burst_thr': THR[a],
                     'median_sigma5_bp': 1e4 * float(np.nanmedian(INP[a]['sig']))} for a, m in MIN.items()])
cov.to_csv(OUT / '01_coverage.csv', index=False); display(cov)

## 2. Real-data trades per split, summaries by period and venue

In [ ]:
T0 = time.time(); TR, ROWS, PA = {}, [], []
for sp in jf.SPLITS:
    TR[sp] = jf.run_split(INP, sp, THR)
    for per in ('dev', 'test'):
        for v in jf.VENUES:
            s, pa = jf.summary(TR[sp], sp, per, v); ROWS.append(s); PA.append(pa)
SUM = pd.DataFrame(ROWS); PA = pd.concat(PA)
SUM.to_csv(OUT / '01_summary.csv', index=False); PA.to_csv(OUT / '01_per_asset.csv', index=False)
REAL = {f'{sp}_{per}': jf.pooled_gross_bp(TR[sp], per) for sp in jf.SPLITS for per in ('dev', 'test')}
pd.concat([t.assign(asset=a) for a, t in TR['all'].items()]).to_csv(OUT / '01_trades_all.csv.gz', index=False, float_format='%.8g')
display(SUM[SUM.venue == jf.PRIMARY_VENUE].round(3)); print(f'{time.time()-T0:.0f}s')

## 3. Sign-flip null (shared signs across assets; full recomputation of σ, splits and the simulation)

In [ ]:
T0 = time.time()
def prog(i, n):
    if i % 20 == 0 or i == n: print(f'  null {i}/{n}  {time.time()-T0:.0f}s', flush=True)
NULL = jf.run_null(MIN, N_NULL, jf.SPLITS, n_jobs=JOBS, progress=prog)
NULL.to_csv(OUT / '01_null.csv', index=False)
display(NULL.describe().T.round(3))

## 4. Verdict

In [ ]:
V = jf.verdict(SUM, NULL, REAL)
V.update({'version': jf.VERSION, 'registered_run': REGISTERED_RUN, 'synthetic': SYNTHETIC, 'n_null': int(len(NULL)), 'real_gross_bp': REAL})
(OUT / '01_decision.json').write_text(json.dumps(V, indent=1, default=str)); print(json.dumps(V, indent=1, default=str))

## 5. Parameter grid (descriptive only; split 'all', Deribit costs)

In [ ]:
G = []
for k in jf.GRID_K:
    for m in jf.GRID_M:
        if m <= k: continue
        for h in jf.GRID_H:
            tr = jf.run_split(INP, 'all', THR, jf.Params(k=k, m=m, h=h))
            for per in ('dev', 'test'):
                s, _ = jf.summary(tr, 'all', per, jf.PRIMARY_VENUE)
                G.append({'K': k, 'M': m, 'H': h, **{x: s[x] for x in ('period', 'trades', 'gross_bp', 'net_bp', 'hit_tp', 'share_stop', 'net_bp_day')}})
G = pd.DataFrame(G); G.to_csv(OUT / '01_grid.csv', index=False)
display(G[G.period == 'test'].pivot_table(index=['K', 'M'], columns='H', values='gross_bp').round(2))
display(G[G.period == 'test'].pivot_table(index=['K', 'M'], columns='H', values='net_bp').round(2))

## 6. Plots: TEST cumulative net (Deribit costs) per split, trade P&L distribution, null vs real

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for sp in jf.SPLITS:
    dn = jf.daily_net({a: jf.add_costs(t, jf.PRIMARY_VENUE) for a, t in TR[sp].items()}, 'test', list(TR[sp]))
    ax[0].plot(dn.index, 1e4 * dn.cumsum(), label=sp)
ax[0].axhline(0, c='k', lw=.5); ax[0].legend(); ax[0].set_title('TEST cumulative net, bp of notional (equal weight)')
z = pd.concat(TR['all'].values()); z = z[z.decision >= jf.DEV_END]
ax[1].hist(1e4 * z.gross.clip(-300, 100), bins=120); ax[1].set_yscale('log'); ax[1].set_title('TEST gross per trade (bp), clipped')
ax[2].hist(NULL['all_test'], bins=30, color='grey'); ax[2].axvline(REAL['all_test'], c='r'); ax[2].set_title('all/TEST gross bp per trade: null vs real')
plt.tight_layout(); plt.savefig(OUT / '01_plots.png', dpi=110); plt.show()